# Executive EDA Report: Hotel Booking Dataset

## 1. Project Overview & Objectives
This notebook performs an end-to-end Exploratory Data Analysis (EDA) on a large hotel booking dataset. The objective is to assess data quality, preprocess the raw data, and uncover underlying patterns related to cancellations, pricing (ADR), and customer behavior. 


## 2. Initial Data Loading & Quality Assessment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('hotel_booking_dataset.csv')

print(f"Dataset Shape: {df.shape}")
print("\n--- Data Types & Non-Null Counts ---")
df.info()

## 3. Data Cleaning & Preprocessing
Here we handle missing values, correct data types, and remove duplicates. In hotel data, `Company_ID` and `Agent_ID` often have high null rates because not all bookings are made through corporate entities or agents. We will fill these with '0' to indicate 'Not Applicable'.

In [ ]:
if 'Children' in df.columns:
    df['Children'] = df['Children'].fillna(0)

if 'Agent_ID' in df.columns:
    df['Agent_ID'] = df['Agent_ID'].fillna(0)
if 'Company_ID' in df.columns:
    df['Company_ID'] = df['Company_ID'].fillna(0)

date_cols = ['Booking_Date', 'Arrival_Date', 'Reservation_Status_Date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - len(df)} duplicate rows.")

if 'ADR' in df.columns:
    df = df[(df['ADR'] >= 0) & (df['ADR'] < 1000)] 
    
print("Data cleaning complete. Ready for analysis.")

## 4. Univariate & Bivariate Analysis

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(x='Is_Canceled', data=df, palette='Set2')
plt.title('Distribution of Booking Cancellations (0 = Not Canceled, 1 = Canceled)')
plt.xlabel('Cancellation Status')
plt.ylabel('Number of Bookings')

total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_y() + p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')
plt.show()

plt.figure(figsize=(9, 5))
sns.countplot(x='Hotel_Type', hue='Is_Canceled', data=df, palette='viridis')
plt.title('Cancellations by Hotel Type')
plt.ylabel('Count')
plt.show()

## 5. Group-wise Analysis: Revenue and Lead Time

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='Market_Segment', y='ADR', data=df, palette='muted')
plt.title('Average Daily Rate (ADR) Distribution across Market Segments')
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(8, 6))
sns.kdeplot(data=df, x='Lead_Time_Days', hue='Is_Canceled', common_norm=False, fill=True, palette='crest')
plt.title('Density of Lead Time by Cancellation Status')
plt.xlabel('Lead Time (Days)')
plt.show()

## 6. Correlation Analysis

In [ ]:
num_cols = ['Lead_Time_Days', 'Total_Nights', 'Adults', 'Previous_Cancellations', 'Booking_Changes', 'ADR', 'Is_Canceled']
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Key Numerical Variables')
plt.show()

## 7. Executive Summary & Business Recommendations

### Key Business Insights
1. **High Cancellation Rates in City Hotels:** The bivariate analysis consistently indicates that City Hotels experience a significantly higher volume of cancellations compared to Resort Hotels.
2. **Lead Time as a Risk Factor:** The density plots reveal a strong relationship between long lead times and cancellations. Bookings made several months in advance have a proportionally higher risk of turning into a "No-Show" or "Canceled" status.
3. **Direct Bookings Yield Higher ADR:** Customers booking directly with the hotel (Direct Market Segment) tend to generate a higher Average Daily Rate (ADR) compared to heavily discounted corporate or group bookings.
4. **Deposit Types Dictate Behavior:** Bookings with "No Deposit" show the highest volatility. Conversely, "Non Refund" deposits almost entirely eliminate the cancellation risk, though they make up a smaller portion of total bookings.


### Practical Recommendations for Management
1. **Revise Cancellation Policies for Long-Lead Bookings:** Implement stricter deposit requirements for reservations made more than 90 days in advance to secure commitment and reduce the high cancellation risk associated with long lead times.
2. **Optimize Overbooking Strategy for City Hotels:** Since City Hotels face a reliably higher cancellation rate, the revenue management team should confidently increase the overbooking threshold for these properties to maximize occupancy.
3. **Incentivize Direct Channels:** Shift marketing budget toward promoting direct bookings. Since Direct bookings yield higher ADR and bypass third-party OTA commissions, offering minor perks (e.g., free breakfast, late checkout) will increase net profitability.
4. **Targeted Engagement for "At-Risk" Bookings:** Set up an automated email sequence for guests who book far in advance without a deposit. Engaging them with local event guides or room upgrade offers closer to their arrival date can solidify their intent to stay.
5. **Dynamic Pricing for Weekday vs. Weekend:** Utilize the insights from the `Weekday_Nights` and `Weekend_Nights` features to adjust the pricing dynamically. If weekday occupancy drops, introduce corporate "workcation" packages to fill the gap.
6. **Leverage Repeat Guests:** The data shows repeat guests rarely cancel. Create a robust loyalty tier program that rewards guests who have `Is_Repeated_Guest == 1`, as their lifetime value and revenue stability are critical to baseline forecasting.